In [ ]:
import os
os.environ['HF_TOKEN'] = "<hf_token>"

# Loading the model from Transformers 

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-3-4b-it"
print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="bfloat16",
  attn_implementation="flash_attention_2" 
)

print("Local model loaded successfully!")
print(f"Parameters: {model.num_parameters():,}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Model size: ~{model.num_parameters() * 2 / 1e9:.1f} GB (bfloat16)")

Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Local model loaded successfully!
Parameters: 4,300,079,472
Vocab size: 262145
Model size: ~8.6 GB (bfloat16)


## Load an SFT Dataset


In [ ]:
system_prompt = "You are a patient that has gone to do an interview with a psychologist. The psychologist will ask you a series of questions and you will answer them in a natural way:\n"
user_prompt = "### Input:\n{question}\n\n### Expected Response:\n{answer}"
def apply_prompt(example):
    # Format each row into a single training text field.
    example["text"] = (
        system_prompt
        + user_prompt.format(question=example["question"], answer=example["answer"])
    )
    return example

In [ ]:
from datasets import load_dataset
import os
print("Loading SFT dataset...")
path =  os.path.dirname(os.path.dirname(os.getcwd()))
path = os.path.join(path, 'data')

train_dataset_sft = load_dataset('json', data_files=path+'/ordered_PT_train_dataset.json').shuffle(seed=42)["train"]
eval_dataset_sft = load_dataset('json', data_files=path+'/ordered_PT_eval_dataset.json').shuffle(seed=42)["train"]
train_dataset_sft = train_dataset_sft.map(apply_prompt)
eval_dataset_sft = eval_dataset_sft.map(apply_prompt)
print("SFT Dataset loaded:")
print(f"    Train samples: {len(train_dataset_sft)}")
print(f"    Eval samples: {len(eval_dataset_sft)}")
print(f"\nSingle Sample: {train_dataset_sft[0]['text']}")

Loading SFT dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/7078 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

SFT Dataset loaded:
   Train samples: 7078
   Eval samples: 750

Single Sample: You are a patient that has gone to do an interview with a psychologist. The psychologist will ask you a series of questions and you will answer them in a natural way:
### Input:
so I'm gonna give them to you one at a time .
 &-um when I give you the picture there's three of them total .
 I just want you to describe as fully to me as you can .
 what's in the picture .
 what you think is going on anything goes here .

### Expected Response:
mhm .


# Train with LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

GLU_MODULES = ["w1", "w2", "w3"]
MHA_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj"]
CONV_MODULES = ["in_proj", "out_proj"]


lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=32,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=GLU_MODULES + MHA_MODULES + CONV_MODULES,
    bias="none",
    modules_to_save=None,
)

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

print("LoRA configuration applied!")
print(f" LoRA rank: {lora_config.r}")
print(f" LoRA alpha: {lora_config.lora_alpha}")
print(f" Target modules: {lora_config.target_modules}")

trainable params: 20,774,912 || all params: 4,320,854,384 || trainable%: 0.4808
LoRA configuration applied!
 LoRA rank: 32
 LoRA alpha: 32
 Target modules: {'w1', 'v_proj', 'k_proj', 'in_proj', 'w3', 'w2', 'out_proj', 'q_proj'}


## Launch Training

Now ready to launch the SFT training, but this time with the LoRA-wrapped model

In [ ]:
from trl import SFTConfig, SFTTrainer

lora_sft_config = SFTConfig(
    output_dir="./lfm2-sft-lora",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    lr_scheduler_type="linear",
    warmup_steps=100,
    warmup_ratio=0.2,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    report_to=None,
)

print("Creating LoRA SFT trainer...")
lora_sft_trainer = SFTTrainer(
    model=lora_model,
    args=lora_sft_config,
    train_dataset=train_dataset_sft,
    eval_dataset=eval_dataset_sft,
    processing_class=tokenizer,
)

print("\nStarting LoRA + SFT training...")
lora_sft_trainer.train()

print("LoRA + SFT training completed!")
merged_model = lora_model.merge_and_unload()
merged_model.push_to_hub("PabloCano1/ordered-PT-gemma3-4b-fine-tuned")
tokenizer.push_to_hub("PabloCano1/ordered-PT-gemma3-4b-fine-tuned")
print(f"LoRA model saved")

Creating LoRA SFT trainer...


Adding EOS to train dataset:   0%|          | 0/7078 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7078 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7078 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

...

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmpp4iqey5y/tokenizer.model      : 100%|##########| 4.69MB / 4.69MB            

  /tmp/tmpp4iqey5y/tokenizer.json       : 100%|##########| 33.4MB / 33.4MB            

LoRA model saved
